[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChiShengChen/neural-signals-101/blob/main/notebooks/zh-TW/13_neuroethics_and_anti_hype.ipynb)

> **在 Google Colab 上執行？** 請先執行下一個儲存格——它會安裝所有套件並取得輔助模組。**在本機執行（執行 `make setup` 之後）？** 下一個儲存格不會做任何事；直接執行並繼續即可。

In [ ]:
# --- Colab 啟動引導：僅在 Colab 環境下安裝相依套件與 neuro101 套件 ---
import sys, os
if "google.colab" in sys.modules:
    !pip install -q "mne==1.8.0" "moabb==1.2.0" "braindecode==0.8.1" "pyriemann==0.7" "scikit-learn==1.5.2"
    if not os.path.exists("neural-signals-101"):
        !git clone -q https://github.com/ChiShengChen/neural-signals-101
    sys.path.insert(0, os.path.abspath("neural-signals-101/src"))
    print("Colab 設定完成——請繼續閱讀下方章節。")

# 第 13 章 — 神經倫理學（Neuroethics）與反炒作（Anti-Hype）

科學新聞媒體熱愛「讀心術」這樣的標題。神經解碼論文受到不成比例的報導，
也受到不成比例的誤導。本章主張：**向公眾過度宣稱，是洩漏（leakage）的倫理學孿生**——
兩者都製造了對技術能力的虛假印象。一個發生在程式碼中；另一個發生在新聞稿裡。

我們將探討*為何*數字會被誇大、在轉譯過程中*遺失了什麼*，
以及持有他人大腦活動資料集的人需要承擔*什麼責任*。

## 學習目標
1. 區分離線（offline）交叉驗證準確率與即時（real-time）、可部署的腦機介面（BCI）效能，
   並闡明為何差距幾乎總是很大。
2. 將常見的「讀心術」標題失敗案例對應到第 12 章的具體陷阱
   （洩漏、樣本數過少（tiny N）、挑選最佳結果（cherry-picking）、無留一受試者（held-out subject））。
3. 描述大腦資料倫理的核心概念：隱私、知情同意（informed consent）、神經權利（neuro-rights）
   與雙重用途風險（dual-use risk）。
4. 總結腦對文字（brain-to-text）／語音神經義肢（speech neuroprostheses）的真實現狀，
   並對比頭皮腦電圖（scalp-EEG）「心靈感應」的宣稱。
5. 將負責任的報告核查清單應用於任何神經解碼結果。

> **前置條件：** 第 10 章與第 12 章。
> **難度：** ★★☆☆☆
> **執行時間：** 約 1 分鐘。

In [ ]:
# 啟動引導：無論是否執行過 `pip install -e .`，都能使 `import neuro101` 正常運作
import sys
from pathlib import Path
try:
    import neuro101  # noqa: F401
except ModuleNotFoundError:
    # 向上搜尋專案根目錄（最多 5 層），找到後將 src 加入路徑
    _p = Path.cwd()
    for _ in range(5):
        if (_p / "src" / "neuro101").exists():
            sys.path.insert(0, str(_p / "src")); break
        _p = _p.parent
    import neuro101  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt
from neuro101 import viz

---
# 第 1 部分 — 「離線 95%」並非一個可運作的腦機介面

## 「離線」是什麼意思？

當研究者報告「95% 準確率」時，他們幾乎總是指**離線（offline）、交叉驗證準確率**：
錄製早已完成，時期（epoch）已被提取，特徵已被計算，
分類器使用該錄製的*留一部分*進行評估。受試者早已離開。
沒有回饋迴路，沒有即時壓力，沒有疲勞，也沒有部署基礎設施。

**即時腦機介面（real-time BCI）** 是完全不同的東西：

| 維度 | 離線評估（Offline evaluation） | 線上／部署腦機介面（Online / deployed BCI） |
|---|---|---|
| 推論（inference）何時發生？ | 錄製完成後 | 使用者嘗試控制裝置時 |
| 信號是否穩定？ | 近似是（單一場次） | 否——分鐘到分鐘都在偏移 |
| 有回饋嗎？ | 無 | 有——使用者會*配合*系統調整 |
| 延遲限制（latency constraint）？ | 無 | 數百毫秒 |
| 校準（calibration）？ | 通常內含在訓練集中 | 每個新場次都必須重新執行 |
| 使用者疲勞 | 無關緊要 | 隨時間增加，信號會偏移 |

## 離線準確率的四大殺手

**1. 非平穩性（Non-stationarity）。** 腦電圖（EEG）信號會漂移。電極阻抗會變化，使用者會移動，
注意力會飄移，而在早上 10 點看起來清晰的神經模式，到 10:15 時可能略有不同。
在場次前半部訓練的模型，若在後半部即時測試，可能面對它從未見過的分布。
我們在**第 12 章，陷阱 5**（跨場次／領域偏移）中具體演示了這一點。

**2. 校準成本（Calibration cost）。** 許多腦機介面系統在能夠使用之前，
需要來自新使用者的幾分鐘有標籤試驗。如果校準困難或令人疲倦，採用率就會崩潰。
「在實驗室中靜坐 20 分鐘後 95%」與「一位從未嘗試過的新使用者在第一次試驗時達到 80%」截然不同。

**3. 使用者疲勞（User fatigue）。** 運動想像腦機介面需要持續的心理努力。
隨著注意力飄移，效能會在一個場次中下降，這就是為何腦機介面研究通常將場次限制在一小時以內。
整齊的、剛收集的資料集上的交叉驗證數字並不包含這種退化。

**4. 延遲（Latency）。** 交叉驗證沒有延遲限制。部署的系統必須即時產生決策——
通常在命令發出後的 100–500 毫秒內。在嵌入式硬體上執行時，
離線「快速」的複雜流水線可能會錯過時間窗。

> **核心見解：** 離線得分 95% 的模型，在即時環境中可能完全無法使用。
> 離線準確率與可用即時效能之間的差距，是腦機介面文獻中最一致、
> 也最被低報的發現之一。

---
# 第 2 部分 — 為何「EEG 讀心術」標題幾乎總是被誇大

讓我們追蹤一個被炒作的結果的典型生命週期。以下每種失敗模式
都與第 12 章的一個具體陷阱相對應。

## 失敗模式 A：受試者洩漏（→ 陷阱 2）

作者招募了 10 名受試者，將他們所有的試驗合併成一個資料集，
隨機將其分成訓練集和測試集，並報告準確率。
因為每個受試者的一些試驗同時出現在訓練集和測試集中，
分類器學習的是*這個人是誰*（他們的個人 EEG 指紋），而非*他們在想什麼*。
這是**第 12 章，陷阱 2**——受試者依賴評估偽裝成泛化。

**誠實的版本**使用留一受試者法（Leave-One-Subject-Out，LOSO）：
測試受試者在訓練期間從未被看到。LOSO 準確率通常低 10–20 個百分點，
特別是對於受試者間變異性很高的頭皮 EEG。

## 失敗模式 B：樣本數過少（→ 陷阱 6）

只有五名受試者且受試者間變異性很高的情況下，LOSO 交叉驗證只有五個折疊。
跨折疊的變異性太大，*最佳*折疊（最配合的受試者）可能比平均值高 20 個百分點。
如果論文報告最佳受試者或最佳種子，那就是**第 12 章，陷阱 6**——挑選幸運的折疊。

負責任的做法是報告**所有折疊／種子的平均值 ± 標準差**，
並明確指出 N = 5 幾乎不足以估計變異性，更遑論做出泛化。

## 失敗模式 C：前處理（Preprocessing）／特徵洩漏（→ 陷阱 3）

特徵選取器、縮放器或 CSP 在交叉驗證迴圈之前，對整個資料集
（所有受試者、所有試驗）進行擬合。「最佳」特徵是那些恰好與完整資料集中的標籤相關的特徵
——包括測試折疊。然後分類器得到了對測試標籤的不公平預覽。
這是**第 12 章，陷阱 3**，它可以從純隨機噪音中製造信號，
正如下面的演示所示。

## 失敗模式 D：無留一受試者；演示（demo）vs 宣稱（claim）

演示不是產品。展示*一個特定的人*（訓練了模型並練習了任務的人）
可以控制一個裝置，是存在性證明，而非可部署性宣稱。
真正的泛化需要一個從未進過實驗室的受試者，
用短暫的場次進行校準，然後達到合理的準確率。
許多「令人印象深刻」的腦機介面演示本質上是一個人執行他們自己的
個性化、高度校準的系統——神經學等效的「在我的機器上可以運行」。

## 為何記者會放大這些失敗

記者不是對手——他們受點擊量激勵，而「腦機介面以 95% 準確率解碼思維」
比「交叉驗證頻帶功率分類器在 9 名受試者的受限想像語音任務中，
以 LOSO 方式達到 0.73 ± 0.12，p = 0.04（多重比較校正前）」獲得更多點擊。
發布誇張新聞稿的科學家對這種通膨負有共同責任。

---
# 第 3 部分 — 資料倫理（Data Ethics）：大腦資料與其他資料不同

## 為何大腦資料值得特殊對待

腦電圖（EEG）錄製是一個電位的時間序列。乍看之下，它像感測器資料——
表面上並不比計步器更敏感。但研究者已經證明，EEG 可以揭示：

- **認知狀態（Cognitive states）**：注意力、工作負荷（workload）、壓力、嗜睡。
- **情緒反應（Emotional responses）**：對產品、人物、政治訊息的喜惡。
- **醫療狀況（Medical conditions）**：癲癇、睡眠障礙、認知衰退的早期跡象。
- **身分（Identity）**：EEG 可作為生物識別標識符——你的大腦節律像指紋一樣獨特。

與密碼不同，你無法更換你的大腦。神經資料的洩露是永久性的。

## 知情同意（Informed Consent）

在學術研究中，機構審查委員會（Institutional Review Board，IRB）或倫理委員會
會審查研究方案。受試者必須以通俗語言被告知：

- 將錄製哪些資料。
- 現在*和*未來將用於什麼用途。
- 誰可以存取。
- 如何儲存、匿名化，並最終刪除。
- 他們可以隨時無懲罰地退出。

機器學習研究者面臨的挑戰是，次要用途——例如，
在合併的腦機介面資料集上訓練基礎模型——可能在原始同意書給予時並未被預見。
即使廣泛的資料共享條款技術上允許，將該資料用於新目的也可能違反原始同意書的精神。

## 神經權利（Neuro-rights）：一個新興框架

幾位倫理學家和法學學者（尤其是 Rafael Yuste 和 Sara Goering）提出了
**神經權利（neuro-rights）**框架——特定於神經領域的基本權利。
核心提案，現已納入智利憲法（2021 年）並在聯合國背景下討論：

| 神經權利 | 保護什麼 |
|---|---|
| **心理隱私（Mental privacy）** | 保持思想私密的權利；未經同意不得掃描或解碼。 |
| **心理完整性（Mental integrity）** | 防止未經授權改變神經活動。 |
| **認知自由（Cognitive liberty）** | 選擇是否增強或擴展自身認知的權利。 |
| **心理連續性（Psychological continuity）** | 保護穩定的自我感免受操控。 |
| **心理增強的平等取得（Equal access to mental augmentation）** | 確保認知增強不會加深不平等。 |

這些是前瞻性的：當前的 EEG 噪音太大，無法讀取詳細的思想。
但法律和倫理框架必須在技術成熟*之前*建立——而非之後。

## 雙重用途風險（Dual-use Risk）

為輪椅使用者或閉鎖症（locked-in）患者建立的腦機介面解碼器是真正的醫療福祉。
**相同的解碼器架構**，若應用於審訊或就業篩選背景下的隱蔽錄製，
就成為監控或強制的工具。這是雙重用途問題，它並非假設性的：

- 情緒偵測耳機已在市場上向雇主出售，用於監控工人的敬業度
  （中國已在工廠和教室中試行）。
- 基於 EEG 的測謊產品存在，儘管缺乏科學驗證，
  並已被用於某些法律程序。
- 軍事研究者資助了具有明顯監視應用的腦機介面研究
  （例如，從無人機操作員的被動 EEG 中進行目標偵測）。

發布開源腦機介面解碼器的研究者無法完全控制其使用方式。
仔細考量下游風險——並在論文中說明——是科學責任的一部分。

> **平衡說明：** 這些都不意味著我們應該停止腦機介面研究。
> 這意味著我們應該在清醒認知下進行，謹慎發表，
> 倡導保護性法規，並建立認真對待這些規範的實踐社群。

---
# 第 4 部分 — 腦對文字（Brain-to-Text）／語音神經義肢（Speech Neuroprostheses）：真實故事

## 真實研究實際做了什麼

2021 年至 2024 年間，幾篇里程碑式論文展示了從**皮質內植入物（intracortical implants）**
對癱瘓受試者嘗試性語音的即時解碼
（例如，Frank Willett 等人 2023 年在《自然》(*Nature*) 上的論文，達到約每分鐘 62 個詞）。
這些是真正令人印象深刻的結果。以下是使其成為可能的因素：

| 因素 | 真實語音神經義肢 | 頭皮 EEG「讀心術」 |
|---|---|---|
| 信號來源 | 皮質內電極陣列（猶他陣列（Utah array）、ECoG 網格）——在神經元層級解析的微伏信號 | 頭皮 EEG——透過顱骨和頭皮擴散的毫伏電位，噪音約高 10,000 倍 |
| 空間解析度（Spatial resolution） | 單單元（single-unit）／小群體層級 | 公分級 |
| 受試者 | 1–5 名高度積極、廣泛校準、同意手術的受試者 | 通常 N < 20 名初次受試者 |
| 校準 | 數百小時的錄製，每位受試者的模型 | 每個場次幾分鐘到幾小時 |
| 模型大小 | 在受試者特定資料上訓練的大型序列模型（RNN、Transformer） | 簡單特徵上的小型分類器 |
| 任務 | 連續的嘗試性語音，完整詞彙表 | 通常是少數幾個詞或命令 |
| 部署 | 具有即時回饋的閉環（closed-loop）系統，測試數月 | 單一錄製場次的離線交叉驗證 |

## 為何差距重要

當記者寫「EEG 以 90% 準確率解碼語音」時，他們幾乎總是在描述這樣的結果：
- 「語音」是 5 個想像的詞。
- 90% 是離線、受試者依賴的準確率。
- 特徵洩漏（第 12 章，陷阱 3）誇大了數字。
- 沒有與機率基線（chance baseline）的比較。
- 沒有人測試過新受試者。

對比真正的神經義肢工作，它對其範圍保持謙遜：
它對*這個特定受試者*有效，使用*這個特定植入物*，
在*這麼多校準*之後，對*這個詞彙表*，
在*留一未見語句上的這個準確率水準*。每一個限定詞都很重要。

## 科學進步是真實的——但在頭皮 EEG 上是適度的

公正地說，頭皮 EEG 可以在謹慎、受控的條件下以適度的準確率對少數心理狀態進行分類。
說它「閱讀思想」是不公正的。這個區別不是學術上的咬文嚼字——
它對政策、資金、患者期望以及公眾對科學的信任都很重要。

---
# 演示——執行前：先思考

我們即將生成**純隨機噪音**作為「EEG 特徵」，並分配**完全隨機的二元標籤**。
從構造上看，特徵和標籤之間的關係為零。

**執行前：預測每種方法會報告什麼。**

- **WRONG（錯誤）** 方法使用*全部*資料（有洩漏）選取 30 個「最佳」特徵，然後進行交叉驗證。
  你預期準確率是多少？
- **RIGHT（正確）** 方法將特徵選取放入流水線（pipeline）中，
  因此選取只在每個折疊中看到訓練資料。你預期準確率是多少？

在這裡寫下你的預測（或只是在心中想一下）：
- 有洩漏的方法：____%
- 誠實的方法：____%

然後執行儲存格，看看會發生什麼。

In [ ]:
# --- 演示：洩漏製造了虛假的「讀心術」結果 ---
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)

# 純隨機噪音——沒有真實的大腦信號，標籤與特徵沒有關聯。
X = rng.standard_normal((200, 2000))   # 200 個「試驗」，2000 個無意義的特徵
y = rng.integers(0, 2, 200)            # 隨機二元標籤

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- WRONG（錯誤）：對全部資料擬合特徵選取器，然後進行交叉驗證 ---
# SelectKBest 在選取「最佳」特徵時，看到了測試折疊的標籤。
selector_leaky = SelectKBest(f_classif, k=30)
X_selected_leaky = selector_leaky.fit(X, y).transform(X)  # 偷看了全部標籤！

acc_leaky = np.mean([
    cross_val_score(
        LogisticRegression(max_iter=500),
        X_selected_leaky, y,
        cv=StratifiedKFold(5, shuffle=True, random_state=s),
    ).mean()
    for s in (0, 1, 2)
])

# --- RIGHT（正確）：特徵選取放在 Pipeline 內（每個折疊單獨擬合）---
pipe_honest = Pipeline([
    ("sel", SelectKBest(f_classif, k=30)),
    ("clf", LogisticRegression(max_iter=500)),
])

acc_honest = np.mean([
    cross_val_score(
        pipe_honest, X, y,
        cv=StratifiedKFold(5, shuffle=True, random_state=s),
    ).mean()
    for s in (0, 1, 2)
])

print(f"資料：{X.shape[0]} 個樣本，{X.shape[1]} 個隨機特徵，隨機標籤。")
print(f"機率基線（chance level）= 0.50")
print()
print(f"WRONG（有洩漏的特徵選取）：準確率 = {acc_leaky:.3f}  <-- 純噪音！")
print(f"RIGHT（流水線內選取）：準確率 = {acc_honest:.3f}  <-- 正確地回到機率水準")
print()
print("「WRONG」的數字正是被炒作的「我們以 X% 解碼了思維！」結果常常呈現的樣子：")
print("是洩漏，不是心靈感應（telepathy）。")

In [ ]:
# --- 圖表：兩欄的「標題 vs 現實」 ---
fig, ax = plt.subplots(figsize=(5.5, 4.5))

viz.plot_wrong_vs_right(
    wrong_score=acc_leaky,
    right_score=acc_honest,
    chance=0.5,
    metric="Accuracy",
    wrong_label="Leaky method\n('mind-reading!')",
    right_label="Honest method\n(pure chance)",
    title="洩漏從噪音中製造出「讀心術」",
    ax=ax,
)
ax.text(
    0.5, -0.13,
    "這就是被炒作的「我們以 90% 解碼了思維！」結果\n通常的樣子——是洩漏，不是心靈感應。",
    ha="center", va="top", transform=ax.transAxes,
    fontsize=8.5, style="italic", color="#555555",
)
plt.show()

---
# 第 5 部分 — 負責任的報告核查清單

複製到你的下一個專案。每個項目都有其原因。

```text
研究設計（STUDY DESIGN）
[ ] 在顯著位置說明受試者數量（N）。
[ ] 說明評估是受試者依賴（subject-DEPENDENT）還是受試者獨立（subject-INDEPENDENT，LOSO）。
    如果是受試者依賴，請標示為此——不要將其作為泛化宣稱呈現。
[ ] 至少包含一個完全留一的測試受試者，其未被用於任何調整或模型選擇決策。

指標（METRICS）
[ ] 報告機率基線（chance-level baseline，例如二元問題為 0.5，k 類問題為 1/k）。
[ ] 報告跨折疊／種子的平均值 ± 標準差——永遠不要只報告單一數字。
[ ] 對不平衡問題使用平衡準確率／F1／ROC-AUC（不只是準確率）。
[ ] 如果比較兩個模型，使用跨折疊的配對檢定（不是對兩個純量的 t 檢定）。

評估完整性（EVALUATION INTEGRITY）
[ ] 每個學習型轉換（縮放器、特徵選取、CSP、ICA、PCA）都在
    Pipeline 內，並僅在訓練折疊上擬合。
[ ] 超參數在驗證集或巢狀 CV 上調整——不在測試集上。
[ ] 區分離線交叉驗證數字與任何線上／即時演示結果。
[ ] 如果你執行了即時演示，同時報告離線 CV 分數和線上分數。

倫理與同意（ETHICS & CONSENT）
[ ] 受試者對其資料的這一具體用途給予了書面知情同意。
[ ] 次要用途（共享、合併、訓練更大模型）已在同意書中涵蓋。
[ ] 資料根據 IRB／倫理委員會要求安全儲存並匿名化。
[ ] 如果該技術可能被濫用，論文中討論了雙重用途風險。

溝通（COMMUNICATION）
[ ] 不從特定任務的結果過度泛化（例如，5 個想像的詞
    ≠「閱讀思想」）。
[ ] 不將實驗室演示作為可部署的產品呈現。
[ ] 發布程式碼（如果同意允許，也發布資料），讓他人可以驗證你的數字。
[ ] 在新聞稿發布前提出審查——在標題成為事實之前抓住它。
```

---
## ✅ 概念測驗

繼續之前先完成這些。答案在下方。

**Q1.** 一篇論文報告使用 8 名受試者以 94% 準確率從 EEG 解碼情緒。
評估是對合併資料集的隨機 80/20 訓練測試切分。
說出這幾乎肯定犯下的兩個第 12 章陷阱，並描述每個陷阱如何誇大報告的數字。

**Q2.** 一家新創公司行銷一款「意念轉文字（thought-to-text）」EEG 頭戴裝置，
引用一篇大學論文聲稱「88% 的想像詞語解碼率」。
列出你要評估該數字是否反映真實世界可用性會問的三個問題。

**Q3.** 一位研究者想與社群分享一個龐大、豐富的 EEG 資料集。
原始同意書涵蓋「在我們的實驗室中用於本研究」。
在公開發布資料之前，他們應該採取什麼倫理步驟？

**Q4.** 在上面的演示中，為何有洩漏的流水線在*純隨機資料*上產生高準確率？
為何誠實的流水線正確地回傳約 0.50？

---
**答案：**

**A1.** (1) **受試者洩漏（陷阱 2）：** 合併受試者並隨機切分，
意味著同一受試者的試驗同時出現在訓練集和測試集中，
因此模型可以學習這個人的 EEG 指紋，而非情緒——誇大了對熟悉受試者的準確率。
(2) **幸運種子／無變異數（陷阱 6）：** N = 8 時幾乎沒有有效的「折疊」；
報告單次執行而不報告變異性，隱藏了結果可能是幸運的這一事實。
額外的第三個陷阱：如果任何前處理是在整個資料集上擬合的，則為**特徵洩漏（陷阱 3）**。

**A2.** 以下三者中的任意三個：(1) 88% 是離線還是線上（即時）？
(2) 是受試者獨立（LOSO）還是受試者依賴？
(3) 詞語有幾個／詞彙表有多大？
(4) 機率基線是多少？
(5) 是否在從未提供訓練資料的新使用者身上測試過？
(6) 是否在實驗室外測試過？

**A3.** 重新聯繫受試者取得涵蓋公開資料發布的補充同意；
如果無法做到，則謹慎地匿名化（刪除識別元資料，
考慮 EEG 指紋識別是否可能重新識別身分）；
諮詢倫理委員會，確認原始的廣泛用途條款是否足夠；
考慮要求對次要使用者進行倫理監督的資料存取協議。

**A4.** 有洩漏的流水線在交叉驗證**之前**對**整個**資料集執行 `SelectKBest.fit(X, y)`。
有了 2000 個隨機特徵，有些會純粹偶然地與隨機標籤相關。
通過恰好選取那 30 個「贏家」，我們交給分類器的特徵與測試折疊標籤存在虛假的對齊——
模型看起來是在泛化，實際上是在記憶統計噪音。
誠實的流水線只在每個折疊的訓練部分擬合 `SelectKBest`，
因此沒有測試標籤的資訊洩漏進來；被選取的特徵對測試集確實沒有資訊價值，
準確率正確地回到約 0.50。

---
## ⚠️ 常見錯誤／為何這樣做是錯的

- **將離線準確率等同於可部署性（deployability）。** 「它在交叉驗證中有效」
  是可用系統的必要條件，但不是充分條件。非平穩性、疲勞、延遲和校準成本
  都會使真實世界效能低於離線數字。部署的腦機介面必須在即時條件下重新驗證。

- **忽略次要資料使用的同意。** 下載公開可用的 EEG 資料集，
  並用它訓練受試者從未同意的用途的模型，即使在技術上合法，
  在倫理上也是有問題的。始終檢查同意範圍。

- **從單一受試者（或高度配合的受試者群）過度泛化。** 在研究者本身或
  從神經科學實驗室招募的志願者身上演示的系統，幾乎肯定會在初次接觸的
  一般人群上表現更差。「對我們有效」與「對所有人有效」之間的差距，
  正是大多數腦機介面產品失敗的地方。

- **將演示視為產品。** 在會議展台上的一個引人注目的即時演示——
  有校準過的使用者、乾淨的錄製環境和精心選擇的任務——
  與產品相距甚遠。從演示到產品的誠實道路
  涉及在混亂環境中對新使用者進行嚴格的留一驗證。

- **將神經權利視為科幻小說而不予理會。** 法律框架有意地走在當前技術*之前*。
  現在——在技術成熟之前——建立倫理規範，比事後加以改造要容易得多。

- **假設雙重用途是別人的問題。** 如果你發表了一種方法，你就是其使用方式
  責任鏈的一部分。這並不意味著你必須拒絕發表；
  它意味著你應該仔細思考、記錄風險，並參與政策討論。

---

**下一步：** 第 14 章——終章。你將在一個你從未見過的留一測試集上
建立自己的完整流水線，遵循第 12 章和第 13 章的每一個核查清單。
評分器將檢查洩漏情況、適當的變異數報告、機率基線，
以及對線上與離線的誠實討論。